<a href="https://colab.research.google.com/github/gredy/2021Z-DataVisualizationTechniques/blob/master/transportation_and_location.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gurobipy
import gurobipy as gp
from gurobipy import GRB

hubs = ["Hialeah", "Doral", "MiamiBeach"]
zones = ["Downtown", "Airport", "Kendall", "MiamiBeach"]

F = {"Hialeah":18000, "Doral":20000, "MiamiBeach":17000}
Cap = {"Hialeah":1200, "Doral":1000, "MiamiBeach":800}
d = {"Downtown":500, "Airport":400, "Kendall":600, "MiamiBeach":300}

c = {
("Hialeah","Downtown"):12, ("Hialeah","Airport"):9,  ("Hialeah","Kendall"):14, ("Hialeah","MiamiBeach"):16,
("Doral","Downtown"):11,   ("Doral","Airport"):8,    ("Doral","Kendall"):15,   ("Doral","MiamiBeach"):17,
("MiamiBeach","Downtown"):10, ("MiamiBeach","Airport"):13, ("MiamiBeach","Kendall"):18, ("MiamiBeach","MiamiBeach"):7
}

m = gp.Model("CFLP")

y = m.addVars(hubs, vtype=GRB.BINARY, name="open")
x = m.addVars(hubs, zones, lb=0, name="ship")

# Objective
m.setObjective(
    gp.quicksum(F[i]*y[i] for i in hubs) +
    gp.quicksum(c[i,j]*x[i,j] for i in hubs for j in zones),
    GRB.MINIMIZE
)

# Demand
for j in zones:
    m.addConstr(gp.quicksum(x[i,j] for i in hubs) == d[j], name=f"demand_{j}")

# Capacity
for i in hubs:
    m.addConstr(gp.quicksum(x[i,j] for j in zones) <= Cap[i]*y[i], name=f"cap_{i}")

m.optimize()

print("Total cost:", m.objVal)
print("Open hubs:", [i for i in hubs if y[i].x > 0.5])
for i in hubs:
    for j in zones:
        if x[i,j].x > 1e-6:
            print(f"{i} -> {j}: {x[i,j].x:.0f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 68.8 MB/s eta 0:00:00
Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 15 columns and 27 nonzeros (Min)
Model fingerprint: 0x503aaa18
Model has 15 linear objective coefficients
Variable types: 12 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [7e+00, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+02, 6e+02]

Presolve time: 0.00s
Presolved: 7 rows, 15 columns, 27 nonzeros
Variable types: 12 continuous, 3 integer (3 binary)
Found heuristic solution: objective 73700.000000

Root relaxation: objective 5.017500e+04, 6 iterations, 0.00 seconds (0.00 work units)

    Nodes    |